# 05 — Streaming: `message/stream` over Server-Sent Events

## Why this notebook exists

In **notebook 04** the client polled `tasks/get` to learn about state changes. Every polling interval that didn't produce news was wasted work, and the client always lagged the server by up to one interval. With many concurrent users on slow tasks, that adds up to a lot of useless requests.

A2A's answer is `message/stream`. Same semantics as `message/send` — *"please do this thing"* — but the server keeps the HTTP connection open and **pushes** events as the work progresses: status updates, artifacts, and a final status that closes the stream.

This notebook builds a streaming researcher, walks through the wire format (Server-Sent Events wrapping JSON-RPC responses), and consumes the stream from a client with no polling at all.

> *Targets A2A spec v0.3.0.*

## What you'll learn

- The Server-Sent Events (SSE) wire format and the `text/event-stream` content type.
- How A2A wraps each SSE event as a complete JSON-RPC 2.0 response.
- The two event kinds: **`TaskStatusUpdateEvent`** (state transitions, with `final: true` on the last one) and **`TaskArtifactUpdateEvent`** (output deliveries).
- How to implement the server side with FastAPI's `StreamingResponse` and a generator.
- How to consume the stream on the client side with `httpx.stream()`.
- Why streaming is strictly better than polling for any non-trivial task — and the one tradeoff (held-open connections) that motivates push notifications in notebook 06.

## 1. Setup

Same helpers as previous notebooks.

In [ ]:
import json
import threading
import time
import uuid
from datetime import datetime, timezone
from typing import Generator, Literal

import httpx
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field, ValidationError, model_validator

_servers: list[uvicorn.Server] = []


def run_server_in_thread(app: FastAPI, port: int) -> uvicorn.Server:
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    for _ in range(50):
        if server.started:
            break
        time.sleep(0.05)
    else:
        raise RuntimeError(f"Server on port {port} did not start in time")
    _servers.append(server)
    return server


def shutdown_all_servers() -> None:
    for server in list(_servers):
        server.should_exit = True
    _servers.clear()


def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


print("Setup OK")

## 2. SSE in 60 seconds

**Server-Sent Events** is a long-lived HTTP response whose body is a series of named events. It's a W3C standard from 2009, supported natively by browsers and by every HTTP client worth using.

Wire format:

```
HTTP/1.1 200 OK
Content-Type: text/event-stream

data: {"some": "json"}

data: {"another": "event"}

data: {"final": "event"}
```

Each event is one or more lines starting with a field name (`data:`, `event:`, `id:`, `retry:`), terminated by a blank line. For A2A we only use `data:` — every event is JSON on a single line.

A2A wraps **each** event as a complete JSON-RPC 2.0 response. So the `data:` payload of every SSE event in an A2A stream has the shape:

```json
{"jsonrpc": "2.0", "id": <request id>, "result": {<event object>}}
```

That means three things:

1. The JSON-RPC `id` is the same on every event of a single stream (the original request's `id`).
2. There is no envelope-level "stream complete" message — instead, the **last** event is a `TaskStatusUpdateEvent` with `final: true`.
3. SSE provides reconnection semantics (the `id:` line, plus the `Last-Event-ID` header on reconnect). A2A inherits all of that for free; we just don't use it in this notebook.

## 3. Event Models

Models carry forward from previous notebooks (`TextPart`, `Message`, `TaskStatus`, `Artifact`, plus the JSON-RPC envelope). We add the two A2A streaming event types.

- **`TaskStatusUpdateEvent`** — fired on every status transition. `kind` is the literal `"taskStatusUpdateEvent"`. The `final: true` flag marks the terminal event that closes the stream.
- **`TaskArtifactUpdateEvent`** — fired when the agent produces an artifact (or a chunk of one). `kind` is `"taskArtifactUpdateEvent"`. We use the simple "one artifact, all at once" pattern; the spec also supports chunked artifacts via optional `append` / `lastChunk` flags, which we won't exercise here.

In [ ]:
TaskState = Literal[
    "submitted", "working", "input-required",
    "completed", "canceled", "failed", "rejected", "auth-required", "unknown",
]


class TextPart(BaseModel):
    kind: Literal["text"] = "text"
    text: str
    metadata: dict | None = None


class Message(BaseModel):
    messageId: str
    role: Literal["user", "agent"]
    parts: list[TextPart] = Field(..., min_length=1)
    kind: Literal["message"] = "message"
    taskId: str | None = None
    contextId: str | None = None
    metadata: dict | None = None


class TaskStatus(BaseModel):
    state: TaskState
    timestamp: str
    message: Message | None = None


class Artifact(BaseModel):
    artifactId: str
    name: str | None = None
    description: str | None = None
    parts: list[TextPart]
    metadata: dict | None = None


class TaskStatusUpdateEvent(BaseModel):
    kind: Literal["taskStatusUpdateEvent"] = "taskStatusUpdateEvent"
    taskId: str
    contextId: str
    status: TaskStatus
    final: bool = False
    metadata: dict | None = None


class TaskArtifactUpdateEvent(BaseModel):
    kind: Literal["taskArtifactUpdateEvent"] = "taskArtifactUpdateEvent"
    taskId: str
    contextId: str
    artifact: Artifact
    append: bool | None = None
    lastChunk: bool | None = None
    metadata: dict | None = None


class JSONRPCRequest(BaseModel):
    jsonrpc: Literal["2.0"] = "2.0"
    id: str | int
    method: str
    params: dict | None = None


class JSONRPCError(BaseModel):
    code: int
    message: str
    data: dict | None = None


class JSONRPCResponse(BaseModel):
    jsonrpc: Literal["2.0"] = "2.0"
    id: str | int | None = None
    result: dict | None = None
    error: JSONRPCError | None = None

    @model_validator(mode="after")
    def _exactly_one(self) -> "JSONRPCResponse":
        if (self.result is None) == (self.error is None):
            raise ValueError("JSONRPCResponse must contain exactly one of `result` or `error`")
        return self


print("Models defined.")

## 4. The Streaming Researcher

The server lives at `POST /`. For `method == "message/stream"` it returns a FastAPI `StreamingResponse` whose body is a generator yielding SSE-formatted bytes. The generator IS the work — `time.sleep` between yields simulates ~1.5s of effort, and each `yield` pushes one event to the client over the open connection.

For `method == "message/send"` we still return a normal one-shot JSON response — same as notebook 03 — so the same agent can serve both flavors. (Real A2A agents typically support both; the Agent Card's `capabilities.streaming` field tells clients which to prefer.)

In [ ]:
FACTS_BY_TOPIC: dict[str, list[str]] = {
    "octopuses": [
        "Octopuses have three hearts.",
        "They can change color in under a second.",
        "Each of their arms has its own neural cluster.",
    ],
    "rome": [
        "Rome was founded in 753 BCE according to tradition.",
        "The Roman Empire at its peak spanned roughly 5 million km².",
        "Roman concrete used volcanic ash and is still studied today.",
    ],
}


researcher_app = FastAPI()


def _sse(rpc_id: str | int, event: BaseModel) -> bytes:
    """Format one A2A event as an SSE `data:` frame."""
    envelope = {
        "jsonrpc": "2.0",
        "id": rpc_id,
        "result": event.model_dump(exclude_none=True),
    }
    return f"data: {json.dumps(envelope)}\n\n".encode("utf-8")


def _stream_research(rpc_id: str | int, topic: str) -> Generator[bytes, None, None]:
    task_id = str(uuid.uuid4())
    context_id = str(uuid.uuid4())

    # 1. Acknowledge: task is submitted (could skip this and go straight to working;
    #    we send it so the learner sees every transition).
    yield _sse(rpc_id, TaskStatusUpdateEvent(
        taskId=task_id, contextId=context_id,
        status=TaskStatus(state="submitted", timestamp=now_iso()),
    ))

    # 2. Working: simulate three steps of effort.
    for step in range(1, 4):
        time.sleep(0.5)
        yield _sse(rpc_id, TaskStatusUpdateEvent(
            taskId=task_id, contextId=context_id,
            status=TaskStatus(
                state="working", timestamp=now_iso(),
                message=Message(
                    messageId=str(uuid.uuid4()), role="agent",
                    parts=[TextPart(text=f"…working on step {step}/3")],
                ),
            ),
        ))

    # 3. Result.
    facts = FACTS_BY_TOPIC.get(topic)
    if facts:
        yield _sse(rpc_id, TaskArtifactUpdateEvent(
            taskId=task_id, contextId=context_id,
            artifact=Artifact(
                artifactId=str(uuid.uuid4()),
                name=f"facts-about-{topic}",
                parts=[TextPart(text=fact) for fact in facts],
            ),
        ))
        yield _sse(rpc_id, TaskStatusUpdateEvent(
            taskId=task_id, contextId=context_id,
            status=TaskStatus(state="completed", timestamp=now_iso()),
            final=True,
        ))
    else:
        yield _sse(rpc_id, TaskStatusUpdateEvent(
            taskId=task_id, contextId=context_id,
            status=TaskStatus(
                state="failed", timestamp=now_iso(),
                message=Message(
                    messageId=str(uuid.uuid4()), role="agent",
                    parts=[TextPart(text=f"No facts on file for {topic!r}.")],
                ),
            ),
            final=True,
        ))


@researcher_app.post("/")
def handle_jsonrpc(req: JSONRPCRequest):
    if req.method != "message/stream":
        # Non-streaming methods unsupported in this notebook; client should use streaming.
        body = {
            "jsonrpc": "2.0",
            "id": req.id,
            "error": {"code": -32601, "message": f"Method not found: {req.method!r}"},
        }
        return body  # FastAPI serializes as application/json

    try:
        msg = Message.model_validate((req.params or {}).get("message"))
    except ValidationError as e:
        body = {
            "jsonrpc": "2.0",
            "id": req.id,
            "error": {"code": -32602, "message": f"Invalid params: {e.errors()}"},
        }
        return body

    topic = msg.parts[0].text.strip().lower()
    return StreamingResponse(
        _stream_research(req.id, topic),
        media_type="text/event-stream",
    )


researcher_server = run_server_in_thread(researcher_app, port=8010)
print("Streaming researcher running on http://127.0.0.1:8010")

## 5. Consuming the Stream

`httpx.stream("POST", ...)` returns a context-managed response we can iterate over **as bytes arrive**. We split lines, look for the `data: ` prefix, and `json.loads` the payload.

Two cells: one prints every event with its kind, then a second one extracts only the final completed-state and artifacts to show the protocol-level result.

In [ ]:
def stream_message(text: str):
    """Yield each parsed event from a message/stream request."""
    envelope = {
        "jsonrpc": "2.0",
        "id": str(uuid.uuid4()),
        "method": "message/stream",
        "params": {
            "message": {
                "messageId": str(uuid.uuid4()),
                "role": "user",
                "parts": [{"kind": "text", "text": text}],
                "kind": "message",
            },
        },
    }
    with httpx.stream(
        "POST", "http://127.0.0.1:8010/", json=envelope, timeout=30.0,
    ) as response:
        response.raise_for_status()
        for line in response.iter_lines():
            if not line.startswith("data: "):
                continue
            payload = json.loads(line[len("data: "):])
            if "error" in payload and payload.get("error") is not None:
                raise RuntimeError(f"JSON-RPC error: {payload['error']}")
            yield payload["result"]


for i, event in enumerate(stream_message("octopuses"), start=1):
    kind = event["kind"]
    if kind == "taskStatusUpdateEvent":
        state = event["status"]["state"]
        final = " (final)" if event.get("final") else ""
        agent_msg = ""
        msg = event["status"].get("message")
        if msg:
            agent_msg = f" — \"{msg['parts'][0]['text']}\""
        print(f"  [{i}] status: {state}{final}{agent_msg}")
    elif kind == "taskArtifactUpdateEvent":
        artifact = event["artifact"]
        print(f"  [{i}] artifact: {artifact['name']} ({len(artifact['parts'])} parts)")
    else:
        print(f"  [{i}] unknown event kind: {kind}")

In [ ]:
def consume_stream(text: str) -> tuple[str, list[Artifact]]:
    """Run a streaming request and collect the final state + artifacts."""
    final_state = "unknown"
    artifacts: list[Artifact] = []
    for event in stream_message(text):
        if event["kind"] == "taskArtifactUpdateEvent":
            artifacts.append(Artifact.model_validate(event["artifact"]))
        elif event["kind"] == "taskStatusUpdateEvent" and event.get("final"):
            final_state = event["status"]["state"]
    return final_state, artifacts


state, artifacts = consume_stream("rome")
print(f"Final state: {state}")
for artifact in artifacts:
    print(f"  artifact: {artifact.name}")
    for part in artifact.parts:
        print(f"    • {part.text}")